# Statistical Analysis: A/B Testing for LLM Evaluation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (8, 4)

## 1. The problem: comparing averages

Model A scores 0.72, Model B scores 0.75. Is B actually better, or is this just noise?

In [ ]:
# Simulate two models that are IDENTICAL (same true accuracy = 0.72)
# Each evaluated on N=20 examples
N = 20
TRUE_ACC = 0.72
N_SIMULATIONS = 10_000

differences = []
for _ in range(N_SIMULATIONS):
    a = rng.binomial(1, TRUE_ACC, N).mean()
    b = rng.binomial(1, TRUE_ACC, N).mean()
    differences.append(b - a)

differences = np.array(differences)

plt.hist(differences, bins=50, edgecolor="white")
plt.axvline(0.03, color="red", linewidth=2, label="Observed diff = 0.03")
plt.xlabel("Score difference (B − A)")
plt.title(f"Distribution of differences when both models are IDENTICAL (N={N})")
plt.legend()
plt.tight_layout()
plt.show()

p_at_least_3pts = np.mean(np.abs(differences) >= 0.03)
print(f"P(|diff| ≥ 0.03) by pure chance = {p_at_least_3pts:.2%}")

## 2. Effect of sample size

The spread of the distribution depends on **N** and **variance**. More samples → narrower distribution → easier to detect real differences.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)

for ax, n in zip(axes, [10, 50, 200]):
    diffs = []
    for _ in range(N_SIMULATIONS):
        a = rng.binomial(1, TRUE_ACC, n).mean()
        b = rng.binomial(1, TRUE_ACC, n).mean()
        diffs.append(b - a)
    ax.hist(diffs, bins=50, edgecolor="white")
    ax.set_title(f"N = {n}  |  std = {np.std(diffs):.3f}")
    ax.set_xlabel("B − A")

plt.suptitle("Same models, different eval set sizes", y=1.02)
plt.tight_layout()
plt.show()

## 3. The t-statistic from scratch

The t-statistic standardises the observed difference by the uncertainty (standard error).  
$$t = \frac{\bar{x}_B - \bar{x}_A}{\text{SE}_{\text{diff}}}$$

In [ ]:
# Generate one concrete sample
n = 40
scores_a = rng.binomial(1, 0.72, n).astype(float)
scores_b = rng.binomial(1, 0.75, n).astype(float)  # B is slightly better

mean_diff = scores_b.mean() - scores_a.mean()
pooled_se = np.sqrt(scores_a.var(ddof=1)/n + scores_b.var(ddof=1)/n)
t_stat    = mean_diff / pooled_se

print(f"Mean A:     {scores_a.mean():.3f}")
print(f"Mean B:     {scores_b.mean():.3f}")
print(f"Difference: {mean_diff:.3f}")
print(f"Pooled SE:  {pooled_se:.3f}")
print(f"t-statistic:{t_stat:.3f}")

In [ ]:
# Visualise the t-distribution and where our t falls
df = 2 * n - 2  # degrees of freedom for two-sample test
x = np.linspace(-4, 4, 400)
y = stats.t.pdf(x, df)

plt.plot(x, y, label=f"t-distribution (df={df})")

# Shade the rejection regions (two-tailed, α=0.05)
crit = stats.t.ppf(0.975, df)
plt.fill_between(x, y, where=np.abs(x) > crit, alpha=0.3, color="red", label=f"Rejection region (α=0.05)")
plt.axvline(t_stat, color="orange", linewidth=2, label=f"Our t = {t_stat:.2f}")

plt.xlabel("t")
plt.legend()
plt.title("Where does our observed t fall under the null hypothesis?")
plt.tight_layout()
plt.show()

p_value = 2 * stats.t.sf(abs(t_stat), df)
print(f"p-value: {p_value:.4f}")
print("Reject H₀ (B = A)?" , "Yes" if p_value < 0.05 else "No")

## 3b. What does the t-boundary mean in score space?

The critical t-value translates into a **minimum detectable difference** in the original score units.  
This makes the rejection threshold concrete: *"with N=50, you need at least X points gap to call it significant."*

In [ ]:
base_score = 0.5   # assume binary scores centred at 0.5 (worst case variance)
alpha = 0.05
ns = np.arange(5, 501, 5)

# For each N: critical t × pooled SE → minimum detectable difference in score units
min_diffs = []
for n in ns:
    t_crit = stats.t.ppf(1 - alpha / 2, df=2 * n - 2)
    se = np.sqrt(2 * base_score * (1 - base_score) / n)  # pooled SE for binary scores
    min_diffs.append(t_crit * se)

min_diffs = np.array(min_diffs)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: zone of indistinguishability in score space
ax1.fill_between(ns, base_score - min_diffs, base_score + min_diffs,
                 alpha=0.3, color="steelblue", label="Indistinguishable zone")
ax1.axhline(base_score, color="black", linestyle="--", linewidth=1, label=f"Baseline = {base_score}")
ax1.set_xlabel("Eval set size (N)")
ax1.set_ylabel("Score")
ax1.set_title("Zone of indistinguishability in score space (α=0.05)")
ax1.set_ylim(0, 1)
ax1.legend()

# Right: minimum detectable gap as a function of N
ax2.plot(ns, min_diffs, color="steelblue")
ax2.axhline(0.05, color="red",    linestyle="--", label="5-point gap")
ax2.axhline(0.10, color="orange", linestyle="--", label="10-point gap")
ax2.set_xlabel("Eval set size (N)")
ax2.set_ylabel("Min detectable difference")
ax2.set_title("Minimum score gap the t-test can detect (α=0.05)")
ax2.legend()

plt.tight_layout()
plt.show()

print("N   | Min detectable difference")
print("----|---------------------------")
for n in [10, 20, 50, 100, 200, 400]:
    t_crit = stats.t.ppf(0.975, df=2 * n - 2)
    se = np.sqrt(2 * base_score * (1 - base_score) / n)
    print(f"{n:4d} | {t_crit * se:.3f}")

## 4b. Choosing the right test

The t-test assumes approximately Gaussian data. LLM evaluations produce many other distributions — pick the test that matches the data generating process.

| Assumed distribution | LLM evaluation example | Standard test | Alternative |
|---|---|---|---|
| **Gaussian** | Average score (continuous judge output) | Welch's t-test | Student's t-test |
| **Binomial** | Binary correctness (pass/fail per example) | Fisher's exact test | Barnard's exact test |
| **Poisson** | Token count / tool calls per response | Poisson E-test | C-test |
| **Multinomial** | Error category counts (factual / incomplete / off-topic) | Chi-squared test | G-test |
| **Unknown** | Human Likert ratings, latency | Mann–Whitney U | Gibbs sampling |

**Fisher's exact test** is particularly relevant here: our binary correctness scores (0/1) from `client.evaluate()` are Binomial, not Gaussian. The t-test is an approximation; Fisher's is exact.

In [ ]:
from scipy.stats import fisher_exact, barnard_exact

# --- Binomial: binary correctness scores ---
# gpt-4o-mini: 15 correct out of 50  |  gpt-4o: 22 correct out of 50
n = 50
correct_mini, correct_4o = 15, 22

# Contingency table: [[correct_A, incorrect_A], [correct_B, incorrect_B]]
table = [
    [correct_mini, n - correct_mini],
    [correct_4o,   n - correct_4o],
]

odds_ratio, p_fisher = fisher_exact(table, alternative="two-sided")
print("=== Binomial: binary correctness (Fisher's exact) ===")
print(f"  gpt-4o-mini: {correct_mini}/{n}  |  gpt-4o: {correct_4o}/{n}")
print(f"  Odds ratio: {odds_ratio:.3f}")
print(f"  p-value:    {p_fisher:.4f}  → {'significant' if p_fisher < 0.05 else 'not significant'}")

# Barnard's test — slightly more powerful for unpaired 2×2 tables (no fixed margins)
res_barnard = barnard_exact(table, alternative="two-sided")
print(f"\n  Barnard's exact p-value: {res_barnard.pvalue:.4f}")
print("  (Barnard does not assume fixed row AND column totals — less conservative)")

In [ ]:
from scipy.stats import chi2_contingency, poisson_means_test, mannwhitneyu

# --- Multinomial: error category counts ---
# How often does each model produce each error type?
#                 factual  incomplete  off-topic
error_counts = np.array([
    [30, 15, 5],   # gpt-4o-mini
    [20, 25, 5],   # gpt-4o
])

chi2, p_chi2, dof, _ = chi2_contingency(error_counts)
G,    p_g,    _,   _ = chi2_contingency(error_counts, lambda_="log-likelihood")

print("=== Multinomial: error category distribution ===")
print(f"  Chi-squared: χ²={chi2:.3f}, p={p_chi2:.4f}, dof={dof}")
print(f"  G-test:       G={G:.3f},   p={p_g:.4f}")
print("  (G-test is preferred when expected counts are small)\n")

# --- Poisson: token counts per response ---
# Total tokens across 50 responses: mini=6000 tokens, 4o=4750 tokens
res_poisson = poisson_means_test(6000, 50, 4750, 50)

print("=== Poisson: token count per response ===")
print(f"  gpt-4o-mini: 6000 tokens / 50 calls = {6000/50:.0f} avg")
print(f"  gpt-4o:      4750 tokens / 50 calls = {4750/50:.0f} avg")
print(f"  Poisson E-test: p={res_poisson.pvalue:.4f}  → {'significant' if res_poisson.pvalue < 0.05 else 'not significant'}\n")

# --- Unknown distribution: human Likert ratings (1–5) ---
ratings_mini = rng.integers(1, 6, 40)   # human scores for model A
ratings_4o   = rng.integers(2, 6, 40)   # human scores for model B (slightly higher)

stat_mw, p_mw = mannwhitneyu(ratings_4o, ratings_mini, alternative="two-sided")

print("=== Unknown distribution: human Likert ratings ===")
print(f"  gpt-4o-mini mean rating: {ratings_mini.mean():.2f}")
print(f"  gpt-4o      mean rating: {ratings_4o.mean():.2f}")
print(f"  Mann-Whitney U: U={stat_mw:.1f}, p={p_mw:.4f}")
print("  (No normality assumption — ranks only)")

## 4. Using scipy — independent and paired t-tests

In [ ]:
# Independent t-test: two separate groups of examples
t, p = stats.ttest_ind(scores_b, scores_a)
print(f"Independent t-test: t={t:.3f}, p={p:.4f}")

# Paired t-test: same examples evaluated by both models
# (reduces variance by removing per-example difficulty)
t_paired, p_paired = stats.ttest_rel(scores_b, scores_a)
print(f"Paired t-test:      t={t_paired:.3f}, p={p_paired:.4f}")

print("\nUse paired when both models answered the same questions.")
print("Paired tests are more powerful — they remove per-question difficulty noise.")

## 5. Power and sample size

**Power** = probability of detecting a real difference when it exists.  
At what N can we reliably detect a 5-point improvement?

In [ ]:
def empirical_power(true_diff, base_acc, n, n_sim=3000, alpha=0.05):
    """Simulate power: fraction of experiments that correctly reject H₀."""
    rejections = 0
    for _ in range(n_sim):
        a = rng.binomial(1, base_acc,            n).astype(float)
        b = rng.binomial(1, base_acc + true_diff, n).astype(float)
        _, p = stats.ttest_ind(b, a)
        if p < alpha:
            rejections += 1
    return rejections / n_sim

sample_sizes = [10, 20, 50, 100, 200, 400]
powers = [empirical_power(true_diff=0.05, base_acc=0.72, n=n) for n in sample_sizes]

plt.plot(sample_sizes, powers, marker="o")
plt.axhline(0.8, color="red", linestyle="--", label="80% power target")
plt.xlabel("Eval set size (N)")
plt.ylabel("Power")
plt.title("Power to detect a 5-point accuracy improvement (α=0.05)")
plt.legend()
plt.tight_layout()
plt.show()

for n, pw in zip(sample_sizes, powers):
    print(f"N={n:4d}  power={pw:.2f}")

## 6. Applying to simulated LLM experiment scores

Standalone simulation that mirrors the LangSmith experiment from notebook 02.

In [ ]:
# Simulate correctness scores from the two models on the same 20 HLE questions
N_EVAL = 20
model_mini_scores = rng.binomial(1, 0.35, N_EVAL).astype(float)  # gpt-4o-mini on PhD questions
model_4o_scores   = rng.binomial(1, 0.45, N_EVAL).astype(float)  # gpt-4o slightly better

print(f"gpt-4o-mini mean: {model_mini_scores.mean():.2f}")
print(f"gpt-4o      mean: {model_4o_scores.mean():.2f}")

t, p = stats.ttest_rel(model_4o_scores, model_mini_scores)
print(f"\nPaired t-test: t={t:.3f}, p={p:.4f}")

if p < 0.05:
    print("→ Significant at α=0.05: gpt-4o appears better on this benchmark.")
else:
    print(f"→ Not significant (p={p:.3f}). With N={N_EVAL} we likely don't have enough power.")
    print(f"   Recall from section 5: you need N≈200 to detect a 10-point difference at 80% power.")

## 8. Multiple comparisons

If you test 10 models at α=0.05, you expect false positives by chance — even when all models are identical.

**Bonferroni** controls the *family-wise error rate* (FWER): probability of *any* false positive. It is conservative — it scales α down by the number of tests.  
**Benjamini-Hochberg (FDR)** controls the *false discovery rate*: the *expected proportion* of rejections that are false. Less conservative — better suited when comparing many models.

In [ ]:
from scipy.stats import false_discovery_control

N_MODELS  = 10
N_EXAMPLES = 50
TRUE_PERF  = 0.60
N_SIM      = 5000
ALPHA      = 0.05

fp_raw        = []
fp_bonferroni = []
fp_fdr        = []

baseline = rng.binomial(1, TRUE_PERF, (N_SIM, N_EXAMPLES)).astype(float)

for sim in range(N_SIM):
    a = baseline[sim]
    p_values = []
    for _ in range(N_MODELS):
        b = rng.binomial(1, TRUE_PERF, N_EXAMPLES).astype(float)
        _, p = stats.ttest_ind(b, a)
        p_values.append(p)

    p_values = np.array(p_values)

    # Uncorrected: any test below α
    fp_raw.append(np.any(p_values < ALPHA))

    # Bonferroni: divide α by number of tests
    fp_bonferroni.append(np.any(p_values < ALPHA / N_MODELS))

    # Benjamini-Hochberg FDR: adjust p-values, then threshold at α
    adjusted = false_discovery_control(p_values, method="bh")
    fp_fdr.append(np.any(adjusted < ALPHA))

print(f"Uncorrected:  {np.mean(fp_raw):.2%}  false positive rate")
print(f"Bonferroni:   {np.mean(fp_bonferroni):.2%}  false positive rate  (controls: P(any FP) ≤ α)")
print(f"BH-FDR:       {np.mean(fp_fdr):.2%}  false positive rate  (controls: E[FP / rejections] ≤ α)")
print()
print(f"Theory: uncorrected ≈ {1 - (1-ALPHA)**N_MODELS:.2%} | Bonferroni ≈ {ALPHA:.2%} | FDR ≈ {ALPHA:.2%} (but less power lost)")

## 8. Multiple comparisons

If you test 10 models at α=0.05, you expect 0.5 false positives by chance — even when all models are identical.

In [ ]:
# Simulate 10 models that are ALL identical
N_MODELS = 10
N_EXAMPLES = 50
TRUE_PERF = 0.60
N_SIM = 5000
ALPHA = 0.05

false_positives_raw        = []
false_positives_bonferroni = []

baseline = rng.binomial(1, TRUE_PERF, (N_SIM, N_EXAMPLES)).astype(float)

for sim in range(N_SIM):
    a = baseline[sim]
    p_values = []
    for _ in range(N_MODELS):
        b = rng.binomial(1, TRUE_PERF, N_EXAMPLES).astype(float)
        _, p = stats.ttest_ind(b, a)
        p_values.append(p)
    
    # Uncorrected: at least one false positive
    false_positives_raw.append(any(p < ALPHA for p in p_values))
    # Bonferroni corrected
    false_positives_bonferroni.append(any(p < ALPHA / N_MODELS for p in p_values))

print(f"Without correction:     {np.mean(false_positives_raw):.2%} of experiments had at least one false positive")
print(f"With Bonferroni (÷{N_MODELS}): {np.mean(false_positives_bonferroni):.2%} of experiments had at least one false positive")
print(f"Expected by theory:     {1 - (1-ALPHA)**N_MODELS:.2%} (uncorrected), ~{ALPHA:.2%} (corrected)")

## Summary: what to report

| Metric | Why |
|---|---|
| Mean ± 95% CI | Central tendency + uncertainty |
| N (eval set size) | Determines power |
| p-value | Probability result is noise |
| Cohen's d (effect size) | Practical significance |
| Latency & token count | Efficiency (from slides: report it!) |

**Saturation caveat**: if all models score >95%, the benchmark tells you nothing useful — a score of 96 vs 97 is noise.

In [ ]:
# Cohen's d for effect size
def cohens_d(a, b):
    pooled_std = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    return (b.mean() - a.mean()) / pooled_std if pooled_std > 0 else 0.0

d = cohens_d(model_mini_scores, model_4o_scores)
t, p = stats.ttest_rel(model_4o_scores, model_mini_scores)
ci_low, ci_high = np.percentile(bootstrap_mean_diff(model_mini_scores, model_4o_scores), [2.5, 97.5])

print("=== Evaluation Report ===")
print(f"Model A (gpt-4o-mini):  mean={model_mini_scores.mean():.3f}  N={N_EVAL}")
print(f"Model B (gpt-4o):       mean={model_4o_scores.mean():.3f}  N={N_EVAL}")
print(f"Difference:             {model_4o_scores.mean() - model_mini_scores.mean():.3f}")
print(f"95% Bootstrap CI:       [{ci_low:.3f}, {ci_high:.3f}]")
print(f"Paired t-test p-value:  {p:.4f}")
print(f"Cohen's d:              {d:.3f}  ({'small' if abs(d)<0.5 else 'medium' if abs(d)<0.8 else 'large'})")